In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
df = spark.read.table('fintech360.business_transformation.bank_data')
df.limit(5).display()

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# 1. CONFIGURATION
# ============================================================
source_table = "fintech360.business_transformation.bank_data"
target_table = "fintech360.gold.bank_data"

spark.sql("CREATE SCHEMA IF NOT EXISTS fintech360.gold")

# ============================================================
# 2. READ BUSINESS-TRANSFORMED DATA
# ============================================================
df = spark.read.table(source_table)

# ============================================================
# 3. DEDUPLICATE INCOMING BATCH
# Keep the latest transformed record for each bank transaction
# ============================================================
dedupe_window = Window.partitionBy("bank_txn_id").orderBy(
    F.col("business_transformed_at").desc_nulls_last(),
    F.col("updated_at").cast("long").desc_nulls_last()
)

df = (
    df.withColumn("_row_num", F.row_number().over(dedupe_window))
      .filter(F.col("_row_num") == 1)
      .drop("_row_num")
)

# ============================================================
# 4. STANDARDIZE CORE BANKING FIELDS
# ============================================================
df = (
    df
    .withColumn("gold_bank_txn_id", F.trim(F.col("bank_txn_id")))
    .withColumn("gold_customer_id", F.col("customer_id").cast("long"))
    .withColumn("gold_account_id", F.trim(F.col("account_id")))
    .withColumn("gold_bank_name", F.upper(F.trim(F.col("business_bank_name"))))
    .withColumn("gold_transaction_type", F.upper(F.trim(F.col("business_transaction_type"))))
    .withColumn("gold_transaction_category", F.upper(F.trim(F.col("business_transaction_category"))))
    .withColumn("gold_currency", F.upper(F.trim(F.col("business_currency"))))
    .withColumn("gold_transaction_amount", F.col("business_transaction_amount").cast("decimal(18,2)"))
    .withColumn("gold_signed_amount", F.col("business_signed_amount").cast("decimal(18,2)"))
    .withColumn("gold_balance_after_txn", F.col("business_balance").cast("decimal(18,2)"))
    .withColumn("gold_narration", F.upper(F.trim(F.col("business_narration"))))
    .withColumn("gold_transaction_direction", F.upper(F.trim(F.col("transaction_direction"))))
)

# ============================================================
# 5. TRANSACTION DATE AND TIME DIMENSIONS
# ============================================================
df = (
    df
    .withColumn(
        "gold_transaction_timestamp",
        F.coalesce(
            F.to_timestamp(F.col("business_transaction_datetime")),
            F.to_timestamp(F.from_unixtime(F.col("transaction_ts").cast("long")))
        )
    )
    .withColumn("gold_transaction_date", F.to_date("gold_transaction_timestamp"))
    .withColumn("gold_transaction_month", F.date_format("gold_transaction_timestamp", "yyyy-MM"))
    .withColumn("gold_transaction_year", F.year("gold_transaction_timestamp"))
    .withColumn("gold_transaction_day", F.dayofmonth("gold_transaction_timestamp"))
    .withColumn("gold_transaction_hour", F.hour("gold_transaction_timestamp"))
    .withColumn("gold_day_of_week", F.date_format("gold_transaction_timestamp", "EEEE"))
    .withColumn(
        "gold_is_weekend",
        F.when(F.dayofweek("gold_transaction_timestamp").isin(1, 7), 1).otherwise(0)
    )
)

# ============================================================
# 6. CASH INFLOW, OUTFLOW AND NET CASH FLOW
# Use the existing business measures where available.
# Fall back to transaction direction and signed amount.
# ============================================================
df = (
    df
    .withColumn(
        "gold_cash_inflow",
        F.coalesce(
            F.col("cash_inflow").cast("double"),
            F.when(
                F.col("gold_transaction_direction") == "INFLOW",
                F.abs(F.col("gold_signed_amount").cast("double"))
            ).otherwise(F.lit(0.0))
        )
    )
    .withColumn(
        "gold_cash_outflow",
        F.coalesce(
            F.col("cash_outflow").cast("double"),
            F.when(
                F.col("gold_transaction_direction") == "OUTFLOW",
                F.abs(F.col("gold_signed_amount").cast("double"))
            ).otherwise(F.lit(0.0))
        )
    )
    .withColumn(
        "gold_net_cash_flow",
        F.coalesce(F.col("gold_cash_inflow"), F.lit(0.0)) -
        F.coalesce(F.col("gold_cash_outflow"), F.lit(0.0))
    )
    .withColumn(
        "gold_is_credit",
        F.when(F.col("gold_transaction_direction") == "INFLOW", 1).otherwise(0)
    )
    .withColumn(
        "gold_is_debit",
        F.when(F.col("gold_transaction_direction") == "OUTFLOW", 1).otherwise(0)
    )
)

# ============================================================
# 7. SALARY SIGNALS
# These indicate salary-like patterns, not verified employment.
# ============================================================
df = (
    df
    .withColumn(
        "gold_salary_signal_flag",
        F.when(
            (F.coalesce(F.col("business_salary_flag").cast("int"), F.lit(0)) == 1) |
            (F.coalesce(F.col("salary_like_credit").cast("int"), F.lit(0)) == 1) |
            (F.coalesce(F.col("salary_narration_flag").cast("int"), F.lit(0)) == 1),
            1
        ).otherwise(0)
    )
    .withColumn(
        "gold_salary_like_credit_amount",
        F.when(
            (F.col("gold_is_credit") == 1) &
            (F.col("gold_salary_signal_flag") == 1),
            F.coalesce(F.col("gold_transaction_amount").cast("double"), F.lit(0.0))
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "gold_salary_signal_status",
        F.when(F.col("gold_salary_signal_flag") == 0, "NO_SALARY_SIGNAL")
         .when(F.col("gold_is_credit") == 1, "SALARY_LIKE_CREDIT_REVIEW")
         .otherwise("SALARY_SIGNAL_DIRECTION_REVIEW")
    )
)

# ============================================================
# 8. TRANSACTION VALUE SEGMENTS
# ============================================================
df = (
    df
    .withColumn(
        "gold_amount_band",
        F.when(F.col("gold_transaction_amount").isNull(), "UNKNOWN")
         .when(F.col("gold_transaction_amount") < 500, "BELOW_500")
         .when(F.col("gold_transaction_amount") < 2000, "500_TO_1999")
         .when(F.col("gold_transaction_amount") < 5000, "2000_TO_4999")
         .when(F.col("gold_transaction_amount") < 10000, "5000_TO_9999")
         .when(F.col("gold_transaction_amount") < 25000, "10000_TO_24999")
         .otherwise("25000_PLUS")
    )
    .withColumn(
        "gold_balance_band",
        F.when(F.col("gold_balance_after_txn").isNull(), "UNKNOWN")
         .when(F.col("gold_balance_after_txn") < 0, "NEGATIVE_BALANCE")
         .when(F.col("gold_balance_after_txn") < 1000, "BELOW_1000")
         .when(F.col("gold_balance_after_txn") < 5000, "1000_TO_4999")
         .when(F.col("gold_balance_after_txn") < 10000, "5000_TO_9999")
         .when(F.col("gold_balance_after_txn") < 25000, "10000_TO_24999")
         .otherwise("25000_PLUS")
    )
    .withColumn(
        "gold_high_value_flag",
        F.when(F.col("gold_transaction_amount") >= 10000, 1).otherwise(0)
    )
    .withColumn(
        "gold_low_balance_flag",
        F.when(
            F.col("gold_balance_after_txn").isNotNull() &
            (F.col("gold_balance_after_txn") < 5000),
            1
        ).otherwise(0)
    )
)

# ============================================================
# 9. NARRATION AND TRANSACTION CLASSIFICATION
# ============================================================
df = (
    df
    .withColumn(
        "gold_narration_group",
        F.when(F.col("gold_narration").rlike("SALARY|PAYROLL|WAGES"), "SALARY_LIKE")
         .when(F.col("gold_narration").rlike("RENT|LANDLORD"), "RENT")
         .when(F.col("gold_narration").rlike("EMI|LOAN REPAYMENT|INSTALLMENT"), "LOAN_REPAYMENT")
         .when(F.col("gold_narration").rlike("UPI|IMPS|NEFT|RTGS|TRANSFER"), "MONEY_TRANSFER")
         .when(F.col("gold_narration").rlike("GROCERY|SUPERMARKET"), "GROCERY")
         .when(F.col("gold_narration").rlike("ELECTRICITY|WATER BILL|MOBILE RECHARGE"), "UTILITIES")
         .when(F.col("gold_narration").isNull(), "UNKNOWN")
         .otherwise("OTHER")
    )
    .withColumn(
        "gold_transaction_review_signal",
        F.when(F.col("gold_bank_txn_id").isNull(), "MISSING_TRANSACTION_ID")
         .when(F.col("gold_customer_id").isNull(), "MISSING_CUSTOMER_ID")
         .when(F.col("gold_account_id").isNull(), "MISSING_ACCOUNT_ID")
         .when(
             F.col("gold_transaction_amount").isNull() |
             (F.col("gold_transaction_amount") < 0),
             "INVALID_TRANSACTION_AMOUNT"
         )
         .when(
             F.col("gold_balance_after_txn").isNotNull() &
             (F.col("gold_balance_after_txn") < 0),
             "NEGATIVE_BALANCE_REVIEW"
         )
         .when(
             (F.col("gold_salary_signal_flag") == 1) &
             (F.col("gold_is_credit") == 0),
             "SALARY_SIGNAL_DIRECTION_REVIEW"
         )
         .when(F.col("gold_high_value_flag") == 1, "HIGH_VALUE_TRANSACTION_REVIEW")
         .when(F.col("gold_low_balance_flag") == 1, "LOW_BALANCE_REVIEW")
         .otherwise("NO_BASIC_REVIEW_SIGNAL")
    )
)

# ============================================================
# 10. DATA QUALITY STATUS AND METADATA
# ============================================================
df = (
    df
    .withColumn(
        "gold_data_quality_status",
        F.when(F.col("gold_bank_txn_id").isNull(), "INVALID")
         .when(F.col("gold_customer_id").isNull(), "INVALID")
         .when(F.col("gold_account_id").isNull(), "INVALID")
         .when(
             F.col("gold_transaction_amount").isNull() |
             (F.col("gold_transaction_amount") < 0),
             "INVALID"
         )
         .otherwise("VALID")
    )
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_table", F.lit(source_table))
)

gold_df = df

# ============================================================
# 11. APPEND TO GOLD TABLE
# Skip transaction IDs already present in the target.
# ============================================================
if spark.catalog.tableExists(target_table):
    existing_ids = (
        spark.read.table(target_table)
        .select(F.col("bank_txn_id").alias("_existing_bank_txn_id"))
        .filter(F.col("_existing_bank_txn_id").isNotNull())
        .dropDuplicates()
    )

    gold_df = gold_df.join(
        existing_ids,
        gold_df["bank_txn_id"] == existing_ids["_existing_bank_txn_id"],
        "left_anti"
    )

gold_df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(target_table)

print(f"Gold data appended successfully to {target_table}")

# ============================================================
# 12. VALIDATE GOLD DATA
# ============================================================
spark.sql(f"""
SELECT
    COUNT(*) AS total_transactions,
    COUNT(DISTINCT bank_txn_id) AS unique_transactions,
    COUNT(DISTINCT customer_id) AS unique_customers,
    COUNT(DISTINCT account_id) AS unique_accounts,
    SUM(gold_cash_inflow) AS total_cash_inflow,
    SUM(gold_cash_outflow) AS total_cash_outflow,
    SUM(gold_net_cash_flow) AS net_cash_flow,
    SUM(gold_salary_signal_flag) AS salary_signal_transactions,
    SUM(gold_high_value_flag) AS high_value_transactions,
    SUM(gold_low_balance_flag) AS low_balance_transactions
FROM {target_table}
""").show(truncate=False)

# Monthly cash flow overview
spark.sql(f"""
SELECT
    gold_transaction_month,
    COUNT(*) AS transaction_count,
    SUM(gold_cash_inflow) AS total_inflow,
    SUM(gold_cash_outflow) AS total_outflow,
    SUM(gold_net_cash_flow) AS net_cash_flow,
    SUM(gold_salary_like_credit_amount) AS salary_like_credit_amount
FROM {target_table}
GROUP BY gold_transaction_month
ORDER BY gold_transaction_month
""").show(100, truncate=False)

# Preview recent records
spark.read.table(target_table) \
    .orderBy(F.col("gold_transaction_timestamp").desc_nulls_last()) \
    .show(10, truncate=False)
